In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import wisc_ecephys_tools as wet

from cnpix_local_sleep.morphological.mua import files
from cnpix_local_sleep.morphological.mua import readers as mua_readers
from cnpix_local_sleep import channel_anatomy
from cnpix_local_sleep import off_tables


In [ ]:
subject = "CNPIX12-Santiago"
experiment = "novel_objects_deprivation"

probe = "imec0"
condition = "Early.REC.NREM"

y_lo = {
    "imec0": 3000,
    "imec1": 5900,
}[probe]
y_hi = {
    "imec0": 7000,
    "imec1": 7200,
}[probe]

off_structures = {"imec0": ["M2", "VO"], "imec1": ["mPPC"]}[probe]

start_time = {
    "Early.REC.NREM": 109102.6259806,  # Page 70
    "Late.NOD.Wake": 107932.335069 - 2.5,
}[condition]
end_time = start_time + 4.0


In [ ]:
if y_lo is None:
    y_lo = 0.0
if y_hi is None:
    y_hi = 384 * 20.0

fig_height = (y_hi - y_lo) / (384 * 20.0) * 8

In [ ]:
s3 = wet.get_sglx_project("shared")
nb = wet.get_sglx_project("shared_nobak")

In [ ]:
show_axis_ticks_and_labels = False
save_plots = True

In [ ]:
def load_mua_raster(smooth: bool) -> xr.DataArray:
    """Load the windowed MUA raster over the [y_lo, y_hi] depth range.

    Substrate is ``mua_traces.zarr`` (500 Hz envelope) loaded directly --
    NOT re-derived from the AP preprocessing chain.

    ``smooth=False``  -> scaled to uV only (raw MUA envelope).
    ``smooth=True``   -> scaled to uV + Gaussian 20 Hz low-pass
        (sigma = fs / (2*pi*20) ~= 4 samples at 500 Hz), i.e. the exact
        detection-time substrate the full-48h morphological thresholds operate on.
    """
    da = mua_readers.open_mua_traces_as_xarray(
        subject,
        probe,
        structure=None,  # all channels; subset by depth below
        condition=None,  # full recording, lazy
        apply_detection_channel_mask=False,
        gaussian_freq_max=20.0 if smooth else None,
    )
    da = da.sel(time=slice(start_time, end_time))  # window BEFORE compute
    da = da.sel(channel=(da.y >= y_lo) & (da.y <= y_hi))
    da = da.sortby("y")  # depth order (ascending y)
    return da.compute()


In [ ]:
from matplotlib.colors import PowerNorm

raster = load_mua_raster(smooth=False)
traces = raster.values  # (time, channel)

# Z-score, just for visualization
traces_mu = traces.mean(axis=0, keepdims=True)
traces_sigma = traces.std(axis=0, keepdims=True)
traces_sigma[traces_sigma == 0] = 1.0
traces = (traces - traces_mu) / traces_sigma

fig, ax = plt.subplots(figsize=(16, fig_height))
ax.imshow(
    np.flipud(traces.T),
    aspect="auto",
    cmap="gist_yarg",
    norm=PowerNorm(gamma=2.0, vmin=None, vmax=np.quantile(traces, 0.95)),
)
if not show_axis_ticks_and_labels:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
if save_plots:
    fig.savefig(
        f"./outputs/fig1_signal/{subject}_{probe}_{'-'.join(off_structures)}_{start_time:.2f}s_morphological_mua.png",
        dpi=300,
        bbox_inches="tight",
    )


### OFF overlay


In [ ]:
# OFF-overlay raster = the detection-time substrate: MUA scaled to uV +
# If `smooth=True`, Gaussian 20 Hz low-pass, exactly what the morphological thresholds saw.
# Disabled here just for visualization of th raw MUA envelope.
snippet_rec = load_mua_raster(smooth=False)
snippet_traces = snippet_rec.values  # (time, channel)

# Z-score, just for visualization
snippet_traces_mu = snippet_traces.mean(axis=0, keepdims=True)
snippet_traces_sigma = snippet_traces.std(axis=0, keepdims=True)
snippet_traces_sigma[snippet_traces_sigma == 0] = 1.0
snippet_traces = (snippet_traces - snippet_traces_mu) / snippet_traces_sigma

snippet_times = snippet_rec.time.values
snippet_fs = snippet_rec.attrs["fs"]
snippet_ycoords = snippet_rec.y.values  # channel depths (ascending)
n_samples, n_channels = snippet_traces.shape


In [ ]:
# Load spike trains for overlay
import cnpix_local_sleep.units as opu

plot_spikes = True

if plot_spikes:
    sorting = opu.load_sorting(subject, probe, unit_quality="all")
    spike_trains = sorting.get_trains_by_property(
        property_name="depth",
        values=snippet_ycoords,
        return_times=True,
        start_time=start_time,
        end_time=end_time,
    )
    # Deduplicate spikes at same depth (multiple units can share a depth)
    spike_trains = {k: np.unique(train) for k, train in sorted(spike_trains.items())}
    print(f"Loaded {len(spike_trains)} spike trains")
    print(f"Total spikes: {sum(len(t) for t in spike_trains.values())}")


In [ ]:
plot_offs = True
overlay_style = "contour"
filter_offs = True
plot_spikes = plot_spikes

# OFF-label source toggle. "full" reads the full-48h detection outputs
# (get_full_*); "per_condition" reads the per-condition spatial outputs
# (matching the tom notebook).
off_source = "full"  # "full" or "per_condition"

q95 = np.quantile(snippet_traces, 0.95)
fig, ax = plt.subplots(figsize=(16, fig_height))
ax.imshow(
    np.flipud(snippet_traces.T),
    aspect="auto",
    cmap="gist_yarg",
    norm=PowerNorm(gamma=2.0, vmin=None, vmax=q95),
)

# structure_colors = dict(zip(off_structures, plt.cm.tab10.colors))
structure_colors = {structure: "orange" for structure in off_structures}

# Time placement is anchored to each OFF's authoritative `start_time` (from
# offs.parquet, computed at detection time as time_coords[start_frame] ==
# time_coords[min(time_ixs)]). Within a single OFF the detection grid is
# uniform at `fs`, so real_time(t_ix) = start_time + (t_ix - min(t_ix)) / fs
# reproduces the detected pixel times exactly.
#
# Deliberately do NOT re-derive the condition-restricted detection grid via
# open_mua_traces_as_xarray(condition=...): the hypnogram mask includes/drops a
# boundary sample inconsistently at each NREM epoch, and those +/-1-sample
# errors accumulate over the recording (measured drift up to ~8 samples /
# ~16 ms by the end of Early.REC.NREM), which would shift the per_condition
# overlay progressively later. Anchoring to start_time is drift-free.

for structure in off_structures:
    if not plot_offs:
        continue
    color = structure_colors[structure]

    # snippet_rec.y is a channel-indexed DataArray of depths -- the exact
    # form compute_channel_mask expects (matches how the reader calls it).
    # Detection channels for a structure are depth-ascending, as is the
    # snippet, so label chan_ixs (0..n_det_chans-1) index det_to_snip_chan.
    mask_chans = channel_anatomy.compute_channel_mask(
        snippet_rec.y, subject, probe, structure, layer=None
    )
    det_to_snip_chan = np.where(np.asarray(mask_chans))[0]
    n_det_chans = len(det_to_snip_chan)

    if off_source == "full":
        offs_path = files.get_full_offs_path(subject, probe, structure)
        lbls_path = files.get_full_off_label_indices_path(subject, probe, structure)
    else:
        offs_path = files.get_offs_path(
            subject=subject,
            probe=probe,
            structure=structure,
            condition=condition,
            threshold_group=None,
        )
        lbls_path = files.get_off_label_indices_path(
            subject=subject,
            probe=probe,
            structure=structure,
            condition=condition,
            threshold_group=None,
        )

    offs = pd.read_parquet(offs_path)
    if filter_offs:
        filters = off_tables.clas_filters
        in_limits = np.full_like(offs.index, True, dtype=bool)
        for col, lims in filters.items():
            in_limits = in_limits & offs[col].between(lims[0], lims[1])
        offs = offs.loc[in_limits]
        offs = offs.reset_index(drop=True)

    offs = offs[(offs["start_time"] >= start_time) & (offs["end_time"] <= end_time)]

    lbls = pd.read_parquet(lbls_path)
    lbls = lbls[lbls["label"].isin(offs["label"])]

    # Accumulate all OFFs for this structure into one imshow-space mask
    # (rows = imshow y = (n_channels - 1) - snippet_chan; cols = snippet sample).
    off_img = np.zeros((n_channels, n_samples), dtype=float)

    for _, off in offs.iterrows():
        lbl_match = lbls[lbls["label"] == off["label"]]
        if lbl_match.empty:
            continue
        lbl = lbl_match.iloc[0]

        t_ixs = np.asarray(lbl["time_ixs"])
        c_ixs = np.asarray(lbl["chan_ixs"])

        # Anchor pixel times to this OFF's authoritative start_time.
        real_times = off["start_time"] + (t_ixs - t_ixs.min()) / snippet_fs

        # Keep pixels inside the snippet window and with in-range channels.
        keep = (
            (real_times >= snippet_times[0])
            & (real_times <= snippet_times[-1])
            & (c_ixs < n_det_chans)
        )
        if not keep.any():
            continue

        snip_x = np.searchsorted(snippet_times, real_times[keep]).clip(0, n_samples - 1)
        snip_chan = det_to_snip_chan[c_ixs[keep]]
        imshow_y = (n_channels - 1) - snip_chan
        off_img[imshow_y, snip_x] = 1.0

    if not off_img.any():
        continue

    # contour/contourf use array indices as coordinates (x=col=snippet sample,
    # y=row=imshow y), matching the imshow above and the spike scatter below.
    if overlay_style == "contour":
        ax.contour(
            off_img,
            levels=[0.5],
            colors=[color],
            linewidths=1.5,
        )
    else:
        ax.contourf(
            off_img,
            levels=[0.5, 1.5],
            colors=[color],
            alpha=0.3,
        )

# Overlay spike trains as tick marks at each unit's depth
if plot_spikes:
    all_spike_x = []
    all_spike_y = []
    for depth, train in spike_trains.items():
        if len(train) == 0:
            continue
        # Convert spike times to snippet sample indices
        spike_samples = np.searchsorted(snippet_times, train).clip(0, n_samples - 1)
        # Find the channel index for this depth
        chan_idx = np.argmin(np.abs(snippet_ycoords - depth))
        # Convert to imshow y coordinate (flipud)
        imshow_y = n_channels - 1 - chan_idx
        all_spike_x.append(spike_samples)
        all_spike_y.append(np.full(len(spike_samples), imshow_y))
    if all_spike_x:
        all_spike_x = np.concatenate(all_spike_x)
        all_spike_y = np.concatenate(all_spike_y)
        ax.scatter(
            all_spike_x,
            all_spike_y,
            s=1,
            c="red",
            alpha=1.0,
            marker="|",
            linewidths=1.5,
        )

if not show_axis_ticks_and_labels:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

if save_plots:
    fig.savefig(
        f"./outputs/fig1_signal/{subject}_{probe}_{'-'.join(off_structures)}_{start_time:.2f}s_morphological_{off_source}_overlay.png",
        dpi=300,
        bbox_inches="tight",
    )


In [ ]:
from cnpix_local_sleep.trace_io import open_lfps
from ecephys.xrsig import core as xrc
from ecephys import plot as eplt

lfps = open_lfps(subject, probe, drop_bad_channels=True)
lfps = lfps.sel(time=slice(start_time - 4, end_time + 4))
lfps = lfps.compute()
lfps = xrc.butter_bandpass(lfps, lowcut=0.5, highcut=300, order=3)
lfps = lfps.sel(time=slice(start_time, end_time))
# lfps = lfps.sel(channel=lfps.y.isin(snippet_ycoords))

In [ ]:
if False:
    fig, ax = plt.subplots(1, 1, figsize=(16, 20))
    eplt.lfp_explorer(
        lfps.time.values, lfps.values, ax, vspace=100, zero_mean=False, flip_dv=True
    )
    ax.axis("off")
    # fig.savefig(nb.get_project_file('figures/nrem_lfp.png'), dpi=600, bbox_inches='tight')
    # fig.savefig(nb.get_project_file('figures/nrem_lfp.svg'), bbox_inches='tight')

In [ ]:
neighborhood_mask = (lfps.y > y_lo - 500) & (lfps.y < y_hi + 500)
lfps_nbr = lfps.sel(channel=neighborhood_mask)

In [ ]:
if True:
    fig, ax = plt.subplots(1, 1, figsize=(16, fig_height))
    vals = lfps_nbr.values[:, :-1:4]
    # vals = lfps_nbr.values[:, ::4]
    eplt.lfp_explorer(
        lfps_nbr.time.values,
        vals,
        ax,
        vspace=200,
        zero_mean=False,
        flip_dv=True,
        # chan_colors=["fuchsia"]  * vals.shape[1]
    )
    ax.axis("off")
    fig.savefig(
        f"./outputs/fig1_signal/{subject}_{probe}_{'-'.join(off_structures)}_{start_time:.2f}s_lfps.png",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )
    # fig.savefig(nb.get_project_file('figures/nrem_lfp.svg'), bbox_inches='tight')

### 3D view of the first OFF period

Renders the first plotted OFF period (only for the `Early.REC.NREM` condition) as a 3D
surface, where the z-scored MUA snippet traces give the altitude. The orange OFF contour
is lifted onto the surface. Spikes are intentionally omitted.


In [ ]:
plot_smooth = False
if condition != "Early.REC.NREM":
    print(
        f"3D view is only rendered for Early.REC.NREM (condition={condition!r}); skipping."
    )
else:
    from scipy.interpolate import RegularGridInterpolator

    if plot_smooth:
        # --- Smoothed altitude: the 20 Hz Gaussian-lowpassed detection substrate ---
        raster_smooth = load_mua_raster(smooth=True)
        smooth_traces = raster_smooth.values  # (time, channel)
        _mu = smooth_traces.mean(axis=0, keepdims=True)
        _sd = smooth_traces.std(axis=0, keepdims=True)
        _sd[_sd == 0] = 1.0
        smooth_traces = (smooth_traces - _mu) / _sd

    # --- Find the first (earliest) OFF period among all plotted structures ---
    # Reuses the exact placement logic from the overlay cell: pixel times are
    # anchored to each OFF's authoritative start_time and mapped into snippet
    # sample / channel indices.
    first = None  # (start_time, structure, color, snip_chan[], snip_x[])
    for structure in off_structures:
        color = structure_colors[structure]
        mask_chans = channel_anatomy.compute_channel_mask(
            snippet_rec.y, subject, probe, structure, layer=None
        )
        det_to_snip_chan = np.where(np.asarray(mask_chans))[0]
        n_det_chans = len(det_to_snip_chan)

        offs_3d = pd.read_parquet(files.get_full_offs_path(subject, probe, structure))
        if filter_offs:
            in_limits = np.full(len(offs_3d), True)
            for col, lims in off_tables.clas_filters.items():
                in_limits = (
                    in_limits & offs_3d[col].between(lims[0], lims[1]).to_numpy()
                )
            offs_3d = offs_3d.loc[in_limits]
        offs_3d = offs_3d[
            (offs_3d["start_time"] >= start_time) & (offs_3d["end_time"] <= end_time)
        ].sort_values("start_time")
        if offs_3d.empty:
            continue

        lbls_3d = pd.read_parquet(
            files.get_full_off_label_indices_path(subject, probe, structure)
        )
        off = offs_3d.iloc[0]
        if first is not None and off["start_time"] >= first[0]:
            continue

        lbl = lbls_3d[lbls_3d["label"] == off["label"]].iloc[0]
        t_ixs = np.asarray(lbl["time_ixs"])
        c_ixs = np.asarray(lbl["chan_ixs"])
        real_times = off["start_time"] + (t_ixs - t_ixs.min()) / snippet_fs
        keep = (
            (real_times >= snippet_times[0])
            & (real_times <= snippet_times[-1])
            & (c_ixs < n_det_chans)
        )
        if not keep.any():
            continue
        snip_x = np.searchsorted(snippet_times, real_times[keep]).clip(0, n_samples - 1)
        snip_chan = det_to_snip_chan[c_ixs[keep]]
        first = (off["start_time"], structure, color, snip_chan, snip_x)

    if first is None:
        print("No OFF period found in the plotted window; skipping 3D view.")
    else:
        _, structure, color, off_snip_chan, off_snip_x = first

        # --- Zoom the surface to a window around the OFF (time + channels) ---
        pad_samples = int(round(0.25 * snippet_fs))
        pad_chans = 6
        x0 = max(int(off_snip_x.min()) - pad_samples, 0)
        x1 = min(int(off_snip_x.max()) + pad_samples + 1, n_samples)
        c0 = max(int(off_snip_chan.min()) - pad_chans, 0)
        c1 = min(int(off_snip_chan.max()) + pad_chans + 1, n_channels)

        xs = np.arange(x0, x1)
        cs = np.arange(c0, c1)
        T, C = np.meshgrid(xs, cs)  # (n_c, n_x) grids
        if plot_smooth:
            Z = smooth_traces[
                x0:x1, c0:c1
            ].T  # altitude = z-scored smoothed MUA (n_c, n_x)
        else:
            Z = snippet_traces[x0:x1, c0:c1].T  # altitude = z-scored MUA (n_c, n_x)

        # OFF mask on the same (channel, sample) grid.
        off_mask = np.zeros((n_channels, n_samples), dtype=float)
        off_mask[off_snip_chan, off_snip_x] = 1.0
        M = off_mask[c0:c1, x0:x1]  # (n_c, n_x)

        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_subplot(111, projection="3d")
        ax.plot_surface(
            T,
            C,
            Z,
            cmap="gist_yarg",
            linewidth=0,
            antialiased=True,
            rcount=min(Z.shape[0], 200),
            ccount=min(Z.shape[1], 200),
            alpha=0.8,
        )

        # --- Lift the orange OFF contour onto the surface ---
        # Extract the 0.5 iso-contour of the mask in 2D, then look up the surface
        # altitude along each contour vertex and draw it slightly above the terrain.
        tmp_fig, tmp_ax = plt.subplots()
        cset = tmp_ax.contour(T, C, M, levels=[0.5])
        segs = [
            poly
            for path in cset.get_paths()
            for poly in path.to_polygons(closed_only=False)
        ]
        plt.close(tmp_fig)

        z_lift = 0.03 * (np.nanmax(Z) - np.nanmin(Z))
        surf_z = RegularGridInterpolator(
            (cs, xs), Z, bounds_error=False, fill_value=None
        )
        for seg in segs:
            if len(seg) == 0:
                continue
            seg_x = seg[:, 0]  # time sample
            seg_c = seg[:, 1]  # channel
            seg_z = surf_z(np.column_stack([seg_c, seg_x])) + z_lift
            ax.plot(seg_x, seg_c, seg_z, color=color, linewidth=2.5)

        ax.set_zlabel("MUA (z-scored)")
        if not show_axis_ticks_and_labels:
            ax.set_xticks([])
            ax.set_yticks([])
        ax.set_xlabel("time" if show_axis_ticks_and_labels else "")
        ax.set_ylabel("channel" if show_axis_ticks_and_labels else "")
        ax.view_init(elev=30, azim=-90)

        if save_plots:
            fig.savefig(
                f"./outputs/fig1_signal/{subject}_{probe}_{structure}_{start_time:.2f}s_morphological_{off_source}_3d.png",
                dpi=300,
                bbox_inches="tight",
            )
